<a href="https://colab.research.google.com/github/sumalya41/QFin.Colab/blob/main/Implied__Volatility_surface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
from scipy.interpolate import griddata, SmoothBivariateSpline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
# numpy scipy jax geomstats matplotlib

Implementing the Black-Scholes Pricing Engine

In [ ]:
def black_scholes_price(S, K, T, r, sigma, option_type='call'):
    """
    Vectorized Black-Scholes option pricing.

    Parameters:
        S: spot price (scalar or array)
        K: strike price (scalar or array)
        T: time to expiration in years
        r: risk-free rate
        sigma: volatility
        option_type: 'call' or 'put'
    """
    if T <= 0:
        return np.maximum(0, S - K) if option_type == 'call' else np.maximum(0, K - S)

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return price

In [ ]:
def implied_volatility(market_price, S, K, T, r, option_type='call'):
    """
    Compute implied volatility using Brent's method.
    Returns NaN if no valid IV exists within [1e-6, 5.0].
    """
    if T <= 0 or market_price <= 0:
        return np.nan

    # Intrinsic value (lower bound for option price)
    intrinsic = max(0, S - K) if option_type == 'call' else max(0, K - S)

    # If market price is below intrinsic, IV is undefined
    if market_price < intrinsic:
        return np.nan

    def objective(sigma):
        return black_scholes_price(S, K, T, r, sigma, option_type) - market_price

    try:
        iv = brentq(objective, 1e-6, 5.0, maxiter=100)
        return iv
    except (ValueError, RuntimeError):
        return np.nan

In [ ]:
# DATA FETCHING
def fetch_option_chain(ticker_symbol="SPY"):
    """
    Fetch option chain data from Yahoo Finance.
    Returns a DataFrame with columns: expiration, strike, option_type,
    bid, ask, mid, implied_volatility.
    """
    ticker = yf.Ticker(ticker_symbol)
    spot_price = ticker.history(period="1d")['Close'].iloc[-1]

    # Get available expiration dates
    expirations = ticker.options

    all_options = []

    for exp_date in expirations[:8]:  # Limit to first 8 expirations for clarity
        opt_chain = ticker.option_chain(exp_date)
        calls = opt_chain.calls
        puts = opt_chain.puts

        # Calculate time to expiration
        T = (pd.to_datetime(exp_date) - pd.Timestamp.now()).days / 365.0
        if T <= 0.01:  # Skip very short-dated options
            continue

        for _, row in calls.iterrows():
            mid_price = (row['bid'] + row['ask']) / 2
            if mid_price > 0.05 and row['strike'] > spot_price * 0.5:
                all_options.append({
                    'expiration': exp_date,
                    'T': T,
                    'strike': row['strike'],
                    'option_type': 'call',
                    'bid': row['bid'],
                    'ask': row['ask'],
                    'mid': mid_price,
                    'volume': row['volume'],
                    'open_interest': row['openInterest']
                })

        for _, row in puts.iterrows():
            mid_price = (row['bid'] + row['ask']) / 2
            if mid_price > 0.05 and row['strike'] < spot_price * 1.5:
                all_options.append({
                    'expiration': exp_date,
                    'T': T,
                    'strike': row['strike'],
                    'option_type': 'put',
                    'bid': row['bid'],
                    'ask': row['ask'],
                    'mid': mid_price,
                    'volume': row['volume'],
                    'open_interest': row['openInterest']
                })

    df = pd.DataFrame(all_options)
    df['spot'] = spot_price
    return df, spot_price

In [ ]:
def build_raw_iv_surface(option_df, r=0.05):
    """
    Calculate implied volatility for each option in the DataFrame.
    Filters out options with very low volume.
    """
    # Filter for liquid options
    df = option_df[option_df['volume'] > 10].copy()

    df['iv'] = df.apply(
        lambda row: implied_volatility(
            row['mid'], row['spot'], row['strike'],
            row['T'], r, row['option_type']
        ), axis=1
    )

    # Remove rows where IV calculation failed
    df = df.dropna(subset=['iv'])

    # Filter unrealistic IV values
    df = df[(df['iv'] > 0.01) & (df['iv'] < 3.0)]

    return df

In [ ]:
# fixed code
def interpolate_surface(iv_df, spot_price):
    """
    Create a smooth implied volatility surface using
    bivariate spline interpolation.

    Returns:
        spline,
        m_grid,
        T_grid,
        M_mesh,
        T_mesh,
        iv_grid
    """

    # -----------------------------
    # Prepare data
    # -----------------------------
    moneyness = iv_df["strike"].values / spot_price
    T = iv_df["T"].values
    iv = iv_df["iv"].values

    # Remove invalid entries
    valid = ~(
        np.isnan(moneyness)
        | np.isnan(T)
        | np.isnan(iv)
    )

    moneyness = moneyness[valid]
    T = T[valid]
    iv = iv[valid]

    # -----------------------------
    # Validate uniqueness
    # -----------------------------
    unique_moneyness = np.unique(moneyness)
    unique_T = np.unique(T)

    if len(unique_moneyness) < 2:
        raise ValueError(
            "Not enough unique moneyness values for spline fitting."
        )

    if len(unique_T) < 2:
        raise ValueError(
            "Not enough unique maturity values for spline fitting."
        )

    # -----------------------------
    # Adaptive spline degrees
    # -----------------------------
    effective_kx = min(3, len(unique_moneyness) - 1)
    effective_ky = min(3, len(unique_T) - 1)

    effective_kx = max(1, effective_kx)
    effective_ky = max(1, effective_ky)

    # -----------------------------
    # Fit spline
    # -----------------------------
    spline = SmoothBivariateSpline(
        x=moneyness,
        y=T,
        z=iv,
        s=0.1,
        kx=effective_kx,
        ky=effective_ky
    )

    # -----------------------------
    # Extract knot domain
    # -----------------------------
    tx, ty = spline.get_knots()

    # IMPORTANT:
    # SmoothBivariateSpline does NOT expose
    # .kx and .ky attributes publicly.
    # Reuse the fitted degrees directly.
    kx = effective_kx
    ky = effective_ky

    m_eval_min = tx[kx]
    m_eval_max = tx[-kx - 1]

    T_eval_min = ty[ky]
    T_eval_max = ty[-ky - 1]

    # -----------------------------
    # Numerical safety buffers
    # -----------------------------
    m_range = m_eval_max - m_eval_min
    T_range = T_eval_max - T_eval_min

    m_eps = (
        min(m_range * 1e-6, m_range * 0.499)
        if m_range > 0
        else 0
    )

    T_eps = (
        min(T_range * 1e-6, T_range * 0.499)
        if T_range > 0
        else 0
    )

    # -----------------------------
    # Visualization grids
    # -----------------------------
    if m_range > 2 * m_eps:
        m_grid = np.linspace(
            m_eval_min + m_eps,
            m_eval_max - m_eps,
            100
        )
    else:
        m_grid = np.full(100, m_eval_min)

    if T_range > 2 * T_eps:
        T_grid = np.linspace(
            T_eval_min + T_eps,
            T_eval_max - T_eps,
            50
        )
    else:
        T_grid = np.full(50, T_eval_min)

    # -----------------------------
    # Mesh generation
    # -----------------------------
    M_mesh, T_mesh = np.meshgrid(
        m_grid,
        T_grid
    )

    # Evaluate surface
    iv_grid = spline(
        M_mesh,
        T_mesh,
        grid=False
    ).reshape(T_mesh.shape)

    return (
        spline,
        m_grid,
        T_grid,
        M_mesh,
        T_mesh,
        iv_grid
    )

In [ ]:
def plot_iv_surface(m_grid, T_grid, M_mesh, T_mesh, iv_grid, spot_price):
    """
    Create publication-quality visualizations of the IV surface.
    """
    fig = plt.figure(figsize=(16, 6))

    # 3D Surface Plot
    ax1 = fig.add_subplot(121, projection='3d')
    surf = ax1.plot_surface(M_mesh, T_mesh, iv_grid, cmap='viridis',
                            edgecolor='none', alpha=0.9, antialiased=True)
    ax1.set_xlabel('Moneyness (K/S)')
    ax1.set_ylabel('Time to Expiration (years)')
    ax1.set_zlabel('Implied Volatility')
    ax1.set_title('Implied Volatility Surface\n3D View')
    ax1.view_init(elev=25, azim=-60)
    fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

    # 2D Contour Heatmap
    ax2 = fig.add_subplot(122)
    contour = ax2.contourf(M_mesh, T_mesh, iv_grid, levels=30, cmap='viridis')
    ax2.set_xlabel('Moneyness (K/S)')
    ax2.set_ylabel('Time to Expiration (years)')
    ax2.set_title('Implied Volatility Surface\nContour Heatmap')
    fig.colorbar(contour, ax=ax2, label='IV')

    plt.tight_layout()
    plt.savefig('volatility_surface.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_cross_sections(spline, m_grid, T_grid, spot_price):
    """
    Plot volatility smile for different maturities and term structure
    for different moneyness levels.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Volatility Smile: IV vs Moneyness for selected maturities
    maturities = [T_grid[len(T_grid)//5], T_grid[len(T_grid)//2], T_grid[-1]]
    colors = ['#2196F3', '#FF9800', '#4CAF50']

    for T_val, color in zip(maturities, colors):
        iv_smile = spline(m_grid, np.full_like(m_grid, T_val), grid=False)
        ax1.plot(m_grid, iv_smile, color=color, linewidth=2,
                label=f'T = {T_val:.2f} years')

    ax1.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='ATM')
    ax1.set_xlabel('Moneyness (K/S)')
    ax1.set_ylabel('Implied Volatility')
    ax1.set_title('Volatility Smile by Maturity')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Term Structure: IV vs Time for selected moneyness levels
    moneyness_levels = [0.9, 1.0, 1.1]
    labels = ['ITM (0.9)', 'ATM (1.0)', 'OTM (1.1)']

    for m_val, label, color in zip(moneyness_levels, labels, colors):
        iv_term = spline(np.full_like(T_grid, m_val), T_grid, grid=False)
        ax2.plot(T_grid, iv_term, color=color, linewidth=2, label=label)

    ax2.set_xlabel('Time to Expiration (years)')
    ax2.set_ylabel('Implied Volatility')
    ax2.set_title('Volatility Term Structure by Moneyness')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('cross_sections.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# fixed main
def main():
    print("Fetching option chain data...")

    option_df, spot_price = fetch_option_chain("SPY")

    print(f"Spot price: ${spot_price:.2f}")
    print(
        f"Fetched {len(option_df)} options across "
        f"{option_df['expiration'].nunique()} expirations"
    )

    print("\nComputing implied volatilities...")
    iv_df = build_raw_iv_surface(option_df)

    print(f"Valid IV points after filtering: {len(iv_df)}")

    print("\nBuilding interpolated surface...")

    spline, m_grid, T_grid, M_mesh, T_mesh, iv_grid = interpolate_surface(
        iv_df,
        spot_price
    )

    print("\nPlotting surfaces...")
    plot_iv_surface(
        m_grid,
        T_grid,
        M_mesh,
        T_mesh,
        iv_grid,
        spot_price
    )

    plot_cross_sections(
        spline,
        m_grid,
        T_grid,
        spot_price
    )

    # Example surface query
    target_moneyness = 1.05   # 5% OTM call
    target_T = 0.25           # 3 months

    iv_query = spline(
        target_moneyness,
        target_T,
        grid=False
    )

    print(
        f"\nIV at {target_moneyness:.2f} moneyness, "
        f"{target_T:.2f}T: {float(iv_query):.2%}"
    )


if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import RBFInterpolator
from mpl_toolkits.mplot3d import Axes3D

# =========================================================
# SAMPLE DATA GENERATOR
# Replace this with your real iv_df later
# =========================================================
def plot_iv_surface(m_grid, T_grid, M_mesh, T_mesh, iv_grid, spot_price):
    """
    Create publication-quality visualizations of the IV surface.
    """
    fig = plt.figure(figsize=(16, 6))

    # 3D Surface Plot
    ax1 = fig.add_subplot(121, projection='3d')
    surf = ax1.plot_surface(M_mesh, T_mesh, iv_grid, cmap='viridis',
                            edgecolor='none', alpha=0.9, antialiased=True)
    ax1.set_xlabel('Moneyness (K/S)')
    ax1.set_ylabel('Time to Expiration (years)')
    ax1.set_zlabel('Implied Volatility')
    ax1.set_title('Implied Volatility Surface\n3D View')
    ax1.view_init(elev=25, azim=-60)
    fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

    # 2D Contour Heatmap
    ax2 = fig.add_subplot(122)
    contour = ax2.contourf(M_mesh, T_mesh, iv_grid, levels=30, cmap='viridis')
    ax2.set_xlabel('Moneyness (K/S)')
    ax2.set_ylabel('Time to Expiration (years)')
    ax2.set_title('Implied Volatility Surface\nContour Heatmap')
    fig.colorbar(contour, ax=ax2, label='IV')

    plt.tight_layout()
    plt.savefig('volatility_surface.png', dpi=150, bbox_inches='tight')
    plt.show()


# =========================================================
# BUILD MANIFOLD
# =========================================================

def build_iv_manifold(iv_df):

    m = iv_df["moneyness"].values
    T = iv_df["T"].values
    iv = iv_df["iv"].values

    X = np.column_stack([m, T])

    manifold = RBFInterpolator(
        X,
        iv,
        kernel="thin_plate_spline",
        smoothing=1e-5
    )

    return manifold


# =========================================================
# GENERATE SURFACE GRID
# =========================================================

def generate_surface(manifold):

    m_grid = np.linspace(0.8, 1.2, 120)
    T_grid = np.linspace(0.02, 2.0, 120)

    M, TT = np.meshgrid(m_grid, T_grid)

    points = np.column_stack([
        M.ravel(),
        TT.ravel()
    ])

    Z = manifold(points).reshape(M.shape)

    return M, TT, Z


# =========================================================
# PLOT MANIFOLD
# =========================================================

def plot_manifold(M, TT, Z):

    fig = plt.figure(figsize=(14,10))

    ax = fig.add_subplot(111, projection='3d')

    surf = ax.plot_surface(
        M,
        TT,
        Z,
        cmap='viridis',
        edgecolor='none',
        alpha=0.95
    )

    ax.set_xlabel("Moneyness")
    ax.set_ylabel("Expiry")
    ax.set_zlabel("Implied Volatility")

    ax.set_title("IV Manifold")

    fig.colorbar(
        surf,
        shrink=0.5,
        aspect=10
    )

    plt.show()


# =========================================================
# CURVATURE
# =========================================================

def gaussian_curvature(Z, dx, dy):

    Zy, Zx = np.gradient(Z, dy, dx)

    Zyy, Zyx = np.gradient(Zy, dy, dx)
    Zxy, Zxx = np.gradient(Zx, dy, dx)

    numerator = (
        Zxx * Zyy -
        Zxy**2
    )

    denominator = (
        1 +
        Zx**2 +
        Zy**2
    )**2

    K = numerator / denominator

    return K


# =========================================================
# CURVATURE HEATMAP
# =========================================================

def plot_curvature(M, TT, K):

    plt.figure(figsize=(12,8))

    plt.contourf(
        M,
        TT,
        K,
        levels=100,
        cmap='coolwarm'
    )

    plt.colorbar(
        label="Gaussian Curvature"
    )

    plt.xlabel("Moneyness")
    plt.ylabel("Expiry")

    plt.title("IV Manifold Curvature")

    plt.show()


# =========================================================
# MAIN
# =========================================================

def main():

    print("Generating synthetic IV data...")

    # The original line 'iv_df = plot_iv_surface()' caused a TypeError.
    # It was attempting to assign the (non-existent) return value of a plotting function
    # to 'iv_df', and was also missing required arguments for that plotting function.
    # Replacing it with synthetic data generation for 'iv_df' as intended by the comment.

    num_points = 100
    synthetic_m = np.random.uniform(0.8, 1.2, num_points)
    synthetic_T = np.random.uniform(0.1, 2.0, num_points)
    # A simple formula for synthetic IV data to create some variation
    synthetic_iv = 0.2 + 0.1 * ((synthetic_m - 1)**2) + 0.05 * synthetic_T

    iv_df = pd.DataFrame({
        "moneyness": synthetic_m,
        "T": synthetic_T,
        "iv": synthetic_iv
    })
    # Define a placeholder spot_price, as it's an argument to plot_iv_surface
    # if it were to be called for actual plotting later.
    spot_price = 100

    print("Building manifold...")

    manifold = build_iv_manifold(iv_df)

    print("Generating surface mesh...")

    M, TT, Z = generate_surface(manifold)

    print("Plotting manifold...")

    plot_manifold(M, TT, Z)

    print("Computing curvature...")

    dx = M[0,1] - M[0,0]
    dy = TT[1,0] - TT[0,0]

    K = gaussian_curvature(Z, dx, dy)

    print("Plotting curvature heatmap...")

    plot_curvature(M, TT, K)

    # If you wish to plot the implied volatility surface using the function
    # defined in this cell, you would call it here with the generated data:
    # print("Plotting generated IV surface...")
    # plot_iv_surface(M, TT, M, TT, Z, spot_price)

    print("Done.")


# =========================================================
# RUN
# =========================================================

main()

In [ ]:
def plot_iv_surface(m_grid, T_grid, M_mesh, T_mesh, iv_grid, spot_price):
    """
    Create publication-quality visualizations of the IV surface.
    """
    fig = plt.figure(figsize=(16, 6))

    # 3D Surface Plot
    ax1 = fig.add_subplot(121, projection='3d')
    surf = ax1.plot_surface(M_mesh, T_mesh, iv_grid, cmap='viridis',
                            edgecolor='none', alpha=0.9, antialiased=True)
    ax1.set_xlabel('Moneyness (K/S)')
    ax1.set_ylabel('Time to Expiration (years)')
    ax1.set_zlabel('Implied Volatility')
    ax1.set_title('Implied Volatility Surface\n3D View')
    ax1.view_init(elev=25, azim=-60)
    fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

    # 2D Contour Heatmap
    ax2 = fig.add_subplot(122)
    contour = ax2.contourf(M_mesh, T_mesh, iv_grid, levels=30, cmap='viridis')
    ax2.set_xlabel('Moneyness (K/S)')
    ax2.set_ylabel('Time to Expiration (years)')
    ax2.set_title('Implied Volatility Surface\nContour Heatmap')
    fig.colorbar(contour, ax=ax2, label='IV')

    plt.tight_layout()
    plt.savefig('volatility_surface.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import SmoothBivariateSpline


# =========================================================
# INTERPOLATE IV SURFACE
# =========================================================

def interpolate_surface(iv_df, spot_price):
    """
    Build smooth implied volatility surface.

    Returns:
        spline
        m_grid
        T_grid
        M_mesh
        T_mesh
        iv_grid
    """

    # -----------------------------------------------------
    # Coordinates
    # -----------------------------------------------------

    moneyness = iv_df["strike"].values / spot_price
    T = iv_df["T"].values
    iv = iv_df["iv"].values

    # -----------------------------------------------------
    # Remove invalid rows
    # -----------------------------------------------------

    valid = (
        np.isfinite(moneyness) &
        np.isfinite(T) &
        np.isfinite(iv)
    )

    moneyness = moneyness[valid]
    T = T[valid]
    iv = iv[valid]

    # -----------------------------------------------------
    # Ensure enough unique points
    # -----------------------------------------------------

    unique_m = np.unique(moneyness)
    unique_T = np.unique(T)

    if len(unique_m) < 4:
        raise ValueError(
            "Need at least 4 unique moneyness points."
        )

    if len(unique_T) < 4:
        raise ValueError(
            "Need at least 4 unique expiry points."
        )

    # -----------------------------------------------------
    # Adaptive spline degrees
    # -----------------------------------------------------

    kx = min(3, len(unique_m) - 1)
    ky = min(3, len(unique_T) - 1)

    kx = max(1, kx)
    ky = max(1, ky)

    # -----------------------------------------------------
    # Fit spline
    # -----------------------------------------------------

    spline = SmoothBivariateSpline(
        x=moneyness,
        y=T,
        z=iv,
        s=0.1,
        kx=kx,
        ky=ky
    )

    # -----------------------------------------------------
    # Evaluation grid
    # -----------------------------------------------------

    m_grid = np.linspace(
        moneyness.min(),
        moneyness.max(),
        120
    )

    T_grid = np.linspace(
        T.min(),
        T.max(),
        80
    )

    M_mesh, T_mesh = np.meshgrid(
        m_grid,
        T_grid
    )

    # -----------------------------------------------------
    # Evaluate surface
    # -----------------------------------------------------

    iv_grid = spline(
        m_grid,
        T_grid,
        grid=True
    ).T

    return (
        spline,
        m_grid,
        T_grid,
        M_mesh,
        T_mesh,
        iv_grid
    )


# =========================================================
# ORIGINAL IV SURFACE PLOT
# =========================================================

def plot_iv_surface(
    m_grid,
    T_grid,
    M_mesh,
    T_mesh,
    iv_grid,
    spot_price
):
    """
    Plot implied volatility surface.
    """

    fig = plt.figure(figsize=(16, 6))

    # -----------------------------------------------------
    # 3D Surface
    # -----------------------------------------------------

    ax1 = fig.add_subplot(
        121,
        projection='3d'
    )

    surf = ax1.plot_surface(
        M_mesh,
        T_mesh,
        iv_grid,
        cmap='viridis',
        edgecolor='none',
        alpha=0.9,
        antialiased=True
    )

    ax1.set_xlabel('Moneyness (K/S)')
    ax1.set_ylabel('Time to Expiration')
    ax1.set_zlabel('Implied Volatility')

    ax1.set_title(
        'Implied Volatility Surface'
    )

    ax1.view_init(
        elev=25,
        azim=-60
    )

    fig.colorbar(
        surf,
        ax=ax1,
        shrink=0.5,
        aspect=10
    )

    # -----------------------------------------------------
    # 2D Contour
    # -----------------------------------------------------

    ax2 = fig.add_subplot(122)

    contour = ax2.contourf(
        M_mesh,
        T_mesh,
        iv_grid,
        levels=30,
        cmap='viridis'
    )

    ax2.set_xlabel(
        'Moneyness (K/S)'
    )

    ax2.set_ylabel(
        'Time to Expiration'
    )

    ax2.set_title(
        'IV Surface Heatmap'
    )

    fig.colorbar(
        contour,
        ax=ax2,
        label='IV'
    )

    plt.tight_layout()

    plt.show()


# =========================================================
# MANIFOLD GEOMETRY
# =========================================================

def gaussian_curvature(
    iv_grid,
    dx,
    dy
):
    """
    Compute Gaussian curvature
    of volatility manifold.
    """

    # -----------------------------------------------------
    # First derivatives
    # -----------------------------------------------------

    dZ_dT, dZ_dm = np.gradient(
        iv_grid,
        dy,
        dx
    )

    # -----------------------------------------------------
    # Second derivatives
    # -----------------------------------------------------

    d2Z_dT2, d2Z_dTdm = np.gradient(
        dZ_dT,
        dy,
        dx
    )

    d2Z_dmdT, d2Z_dm2 = np.gradient(
        dZ_dm,
        dy,
        dx
    )

    # -----------------------------------------------------
    # Gaussian curvature formula
    # -----------------------------------------------------

    numerator = (
        d2Z_dm2 * d2Z_dT2 -
        d2Z_dTdm**2
    )

    denominator = (
        1 +
        dZ_dm**2 +
        dZ_dT**2
    )**2

    K = numerator / denominator

    return K


# =========================================================
# CURVATURE PLOT
# =========================================================

def plot_manifold_curvature(
    M_mesh,
    T_mesh,
    curvature
):
    """
    Plot Gaussian curvature
    of volatility manifold.
    """

    plt.figure(figsize=(12, 8))

    contour = plt.contourf(
        M_mesh,
        T_mesh,
        curvature,
        levels=100,
        cmap='coolwarm'
    )

    plt.xlabel(
        "Moneyness (K/S)"
    )

    plt.ylabel(
        "Time to Expiration"
    )

    plt.title(
        "Volatility Manifold Gaussian Curvature"
    )

    plt.colorbar(
        contour,
        label="Gaussian Curvature"
    )

    plt.show()


# =========================================================
# OPTIONAL:
# TANGENT VECTOR FIELD
# =========================================================

def plot_gradient_field(
    M_mesh,
    T_mesh,
    iv_grid
):
    """
    Plot local manifold gradient field.
    """

    dx = M_mesh[0,1] - M_mesh[0,0]
    dy = T_mesh[1,0] - T_mesh[0,0]

    dZ_dT, dZ_dm = np.gradient(
        iv_grid,
        dy,
        dx
    )

    plt.figure(figsize=(12,8))

    plt.contourf(
        M_mesh,
        T_mesh,
        iv_grid,
        levels=30,
        cmap='viridis'
    )

    plt.quiver(
        M_mesh[::5, ::5],
        T_mesh[::5, ::5],
        dZ_dm[::5, ::5],
        dZ_dT[::5, ::5],
        color='white'
    )

    plt.xlabel("Moneyness")
    plt.ylabel("Expiry")

    plt.title(
        "Volatility Manifold Gradient Field"
    )

    plt.show()


# =========================================================
# MAIN PIPELINE
# =========================================================

def main():

    print("Fetching option chain...")

    option_df, spot_price = fetch_option_chain(
        "SPY"
    )

    print(
        f"Spot Price: {spot_price:.2f}"
    )

    # -----------------------------------------------------
    # Build IV dataset
    # -----------------------------------------------------

    print("Building IV surface...")

    iv_df = build_raw_iv_surface(
        option_df
    )

    print(
        f"Valid IV points: {len(iv_df)}"
    )

    # -----------------------------------------------------
    # Interpolate surface
    # -----------------------------------------------------

    (
        spline,
        m_grid,
        T_grid,
        M_mesh,
        T_mesh,
        iv_grid
    ) = interpolate_surface(
        iv_df,
        spot_price
    )

    # -----------------------------------------------------
    # Original IV surface
    # -----------------------------------------------------

    print(
        "Plotting IV surface..."
    )

    plot_iv_surface(
        m_grid,
        T_grid,
        M_mesh,
        T_mesh,
        iv_grid,
        spot_price
    )

    # =====================================================
    # MANIFOLD ANALYSIS
    # =====================================================

    print(
        "Computing manifold curvature..."
    )

    dx = (
        M_mesh[0,1] -
        M_mesh[0,0]
    )

    dy = (
        T_mesh[1,0] -
        T_mesh[0,0]
    )

    curvature = gaussian_curvature(
        iv_grid,
        dx,
        dy
    )

    # -----------------------------------------------------
    # Curvature topology
    # -----------------------------------------------------

    print(
        "Plotting manifold curvature..."
    )

    plot_manifold_curvature(
        M_mesh,
        T_mesh,
        curvature
    )

    # -----------------------------------------------------
    # Tangent vector field
    # -----------------------------------------------------

    print(
        "Plotting manifold gradient field..."
    )

    plot_gradient_field(
        M_mesh,
        T_mesh,
        iv_grid
    )

    # -----------------------------------------------------
    # Query example
    # -----------------------------------------------------

    target_moneyness = 1.05
    target_T = 0.25

    iv_query = spline(
        target_moneyness,
        target_T,
        grid=False
    )

    print(
        f"\nIV at "
        f"{target_moneyness:.2f} moneyness "
        f"and {target_T:.2f} years = "
        f"{float(iv_query):.2%}"
    )


# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":

    main()

experiements with 3D graphing

In [ ]:
import plotly.graph_objects as go
import numpy as np


def plot_iv_manifold_3d(
    M_mesh,
    T_mesh,
    iv_grid,
    curvature=None
):
    """
    Interactive 3D volatility manifold visualization.
    """

    # -----------------------------------------------------
    # Optional curvature coloring
    # -----------------------------------------------------

    if curvature is None:
        surface_color = iv_grid
        color_title = "Implied Volatility"

    else:
        surface_color = curvature
        color_title = "Gaussian Curvature"

    # -----------------------------------------------------
    # Build surface
    # -----------------------------------------------------

    fig = go.Figure(
        data=[
            go.Surface(
                x=M_mesh,
                y=T_mesh,
                z=iv_grid,

                surfacecolor=surface_color,

                colorscale='Viridis',

                colorbar=dict(
                    title=color_title
                ),

                lighting=dict(
                    ambient=0.6,
                    diffuse=0.9,
                    roughness=0.3,
                    specular=0.4
                ),

                lightposition=dict(
                    x=100,
                    y=200,
                    z=300
                )
            )
        ]
    )

    # -----------------------------------------------------
    # Layout
    # -----------------------------------------------------

    fig.update_layout(

        title="Implied Volatility Manifold",

        width=1200,
        height=900,

        scene=dict(

            xaxis_title="Moneyness",

            yaxis_title="Expiry",

            zaxis_title="Implied Volatility",

            aspectmode='manual',

            aspectratio=dict(
                x=1.2,
                y=1.2,
                z=0.7
            ),

            camera=dict(

                eye=dict(
                    x=1.8,
                    y=1.8,
                    z=1.2
                )
            )
        )
    )

    fig.show()

In [ ]:
def plot_tangent_vectors(
    M_mesh,
    T_mesh,
    iv_grid
):

    import plotly.graph_objects as go

    dx = M_mesh[0,1] - M_mesh[0,0]
    dy = T_mesh[1,0] - T_mesh[0,0]

    dZ_dT, dZ_dm = np.gradient(
        iv_grid,
        dy,
        dx
    )

    step = 6

    fig = go.Figure()

    # -------------------------------------------------
    # Surface
    # -------------------------------------------------

    fig.add_trace(
        go.Surface(
            x=M_mesh,
            y=T_mesh,
            z=iv_grid,
            opacity=0.85,
            colorscale='Viridis'
        )
    )

    # -------------------------------------------------
    # Tangent vectors
    # -------------------------------------------------

    for i in range(0, M_mesh.shape[0], step):
        for j in range(0, M_mesh.shape[1], step):

            x0 = M_mesh[i,j]
            y0 = T_mesh[i,j]
            z0 = iv_grid[i,j]

            u = 0.02
            v = 0.02

            w = (
                dZ_dm[i,j] * u +
                dZ_dT[i,j] * v
            )

            fig.add_trace(
                go.Scatter3d(

                    x=[x0, x0 + u],
                    y=[y0, y0 + v],
                    z=[z0, z0 + w],

                    mode='lines',

                    line=dict(
                        color='white',
                        width=4
                    ),

                    showlegend=False
                )
            )

    fig.update_layout(
        title="Volatility Manifold Tangent Field",
        height=900
    )

    fig.show()

In [ ]:
# The line below attempts to use a GUI backend ('TkAgg') which is not supported in Colab's headless environment.
# In most Colab use cases, you don't need to explicitly set a backend, or can use magic commands like
# %matplotlib inline or %matplotlib notebook for interactive plots directly in the notebook output.
# If you need to generate images without displaying a GUI, 'Agg' is a common non-interactive backend.
# For now, we will comment out this line as it is causing the error.
# import matplotlib
# matplotlib.use("TkAgg")  # or "Qt5Agg"

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Surface(
    x=M_mesh,
    y=T_mesh,
    z=iv_grid,
    colorscale='Viridis'
)])

fig.show()